# Fine-tuning YOLO11n for PPE detection

Same steps as `scripts/train_ppe.py`: build a YOLO-format dataset (synthetic by default, or your own PPE dataset), fine-tune `yolo11n.pt`, evaluate mAP, export to ONNX and drop it into `models/` so the inference service lists it as `ppe-yolo11n-onnx`.

Run with `uv sync --extra yolo && uv run --with jupyter jupyter lab`.

In [ ]:
from pathlib import Path
import sys
ROOT = Path.cwd().parent
sys.path.insert(0, str(ROOT / 'scripts'))
from train_ppe import build_synthetic_dataset, CLASSES
CLASSES

## 1. Dataset
Use the synthetic generator, or set `DATA` to a real dataset yaml whose class names are `person`, `helmet`, `no_helmet` (e.g. a hard-hat dataset exported in YOLOv8 format).

In [ ]:
DATA = build_synthetic_dataset(ROOT / 'datasets' / 'ppe-synthetic')
# DATA = Path('/datasets/hardhat/data.yaml')
print(DATA.read_text())

## 2. Inspect a labelled sample

In [ ]:
import cv2, matplotlib.pyplot as plt
img_path = next((DATA.parent / 'images' / 'train').glob('*.jpg'))
img = cv2.imread(str(img_path))
h, w = img.shape[:2]
for row in (DATA.parent / 'labels' / 'train' / (img_path.stem + '.txt')).read_text().strip().splitlines():
    c, cx, cy, bw, bh = map(float, row.split())
    x1, y1 = int((cx - bw/2) * w), int((cy - bh/2) * h)
    x2, y2 = int((cx + bw/2) * w), int((cy + bh/2) * h)
    cv2.rectangle(img, (x1, y1), (x2, y2), (0, 255, 0), 1)
    cv2.putText(img, CLASSES[int(c)], (x1, y1 - 2), cv2.FONT_HERSHEY_SIMPLEX, 0.3, (0, 255, 0), 1)
plt.figure(figsize=(10, 6)); plt.imshow(img[..., ::-1]); plt.axis('off');

## 3. Fine-tune
CPU works for a few epochs at `imgsz=320`; use a GPU (`device=0`) for real datasets.

In [ ]:
from ultralytics import YOLO
model = YOLO(str(ROOT / 'models' / 'yolo11n.pt'))
model.train(data=str(DATA), epochs=5, imgsz=320, batch=16, device='cpu', project=str(ROOT / 'runs'), name='ppe-nb', exist_ok=True)

## 4. Evaluate

In [ ]:
best = YOLO(str(ROOT / 'runs' / 'ppe-nb' / 'weights' / 'best.pt'))
m = best.val(data=str(DATA), imgsz=320, device='cpu')
print(f'mAP50={m.box.map50:.3f}  mAP50-95={m.box.map:.3f}')
for name, ap in zip(CLASSES, m.box.maps):
    print(f'{name:10s} AP50-95={ap:.3f}')

## 5. Export to ONNX and benchmark

In [ ]:
onnx_path = best.export(format='onnx', imgsz=320, simplify=True)
import shutil, time; shutil.copy(onnx_path, ROOT / 'models' / 'ppe-yolo11n.onnx')
from visionguard_ml.detectors.onnx_yolo import OnnxYoloDetector
det = OnnxYoloDetector(ROOT / 'models' / 'ppe-yolo11n.onnx', 'ppe-yolo11n-onnx')
frame = cv2.imread(str(img_path))
t0 = time.perf_counter(); [det.detect(frame) for _ in range(50)]
print(f'{(time.perf_counter() - t0) / 50 * 1000:.1f} ms / frame', det.detect(frame)[:3])

Restart the inference service and activate **YOLO11n fine-tuned PPE (ONNX)** on the Models page.